In [0]:
-- =========================================================
-- Silver: Patients
-- deduplication, drop personally identifiable information,
-- derive age, age band, decreased flag
-- =========================================================

CREATE OR REPLACE TABLE health_lakehouse.silver.patients AS
WITH deduplicated AS (
    SELECT *, ROW_NUMBER() OVER (PARTITION BY Id ORDER BY _ingested_at DESC) AS rn FROM health_lakehouse.bronze.patients
),
cleaned AS (
    SELECT
    Id AS patient_id,
    BIRTHDATE AS birth_date,
    DEATHDATE AS death_date,
    MARITAL AS marital_status,
    RACE AS race,
    ETHNICITY AS ethnicity,
    GENDER AS gender,
    CITY AS city,
    STATE AS state,
    COUNTY AS county,
    ZIP AS zip,
    LAT AS latitude,
    LON AS longitude,
    HEALTHCARE_EXPENSES AS healthcare_expenses,
    HEALTHCARE_COVERAGE AS healthcare_coverage,
    INCOME AS income,
    -- age at death if deceased, else age today
    CAST(FLOOR(DATEDIFF(COALESCE(DEATHDATE, current_date()),BIRTHDATE)/365.25)AS INT) AS age,
    (DEATHDATE IS NOT NULL) AS is_deceased
    FROM deduplicated
    WHERE rn=1 AND Id IS NOT NULL
),
banded AS (
    SELECT *,
        CASE
            WHEN age < 18 THEN '0-17'
            WHEN age < 35 THEN '18-34'
            WHEN age < 50 THEN '35-49'
            WHEN age < 65 THEN '50-64'
            WHEN age < 80 THEN '65-79'
            ELSE '80+'
        END AS age_band
    FROM cleaned
)
SELECT * FROM banded;


In [0]:
-- =========================================================
-- Silver: encounters
-- deduplication, rename, derive lenght of stay,
-- validate timestamps
-- =========================================================

CREATE OR REPLACE TABLE health_lakehouse.silver.encounters AS
WITH deduplicated AS(
    SELECT *, ROW_NUMBER() OVER (PARTITION BY Id ORDER BY _ingested_at DESC) AS rn
    FROM health_lakehouse.bronze.encounters
),
cleaned AS (
    SELECT
    Id AS encounter_id,
    PATIENT AS patient_id,
    ORGANIZATION AS organization_id,
    PROVIDER AS provider_id,
    PAYER AS payer_id,
    START AS start_ts,
    STOP AS stop_ts,
    ENCOUNTERCLASS AS encounter_class,
    CAST (CODE AS STRING) AS encounter_code,
    DESCRIPTION AS encounter_desc,
    BASE_ENCOUNTER_COST AS base_cost,
    TOTAL_CLAIM_COST as total_calim_cost,
    PAYER_COVERAGE AS payer_coverage,
    CAST(REASONCODE AS STRING) AS reason_code,
    REASONDESCRIPTION AS reason_desc,
    -- lenght of stay in hours (encounters can be less than a day)
    ROUND((UNIX_TIMESTAMP(STOP)-UNIX_TIMESTAMP(START))/3600.0,2) AS los_hours
    FROM deduplicated
    WHERE rn =1
        AND Id IS NOT NULL
        AND PATIENT IS NOT NULL
        AND START IS NOT NULL
)
SELECT * FROM cleaned
WHERE stop_ts IS NOT NULL OR stop_ts >= start_ts; -- to drop impossible time-travel rows


In [0]:
-- checking new tables.
SELECT COUNT(*) AS patient_rows FROM health_lakehouse.silver.patients;
SELECT * FROM health_lakehouse.silver.patients LIMIT 5;

SELECT COUNT(*) AS encounter_rows FROM health_lakehouse.silver.encounters;
SELECT * FROM health_lakehouse.silver.encounters LIMIT 5;

In [0]:
-- =========================================================
-- Silver: conditions
-- No natural primary key (create composite key on depuplicated)
-- Derives:is_active, duration days
-- =========================================================
CREATE OR REPLACE TABLE health_lakehouse.silver.conditions AS
WITH deduplicated AS (
    SELECT *, ROW_NUMBER()OVER(
        PARTITION BY PATIENT, ENCOUNTER, CODE, START
        ORDER BY _ingested_at DESC
    )AS rn
    FROM health_lakehouse.bronze.conditions
),
cleanded AS (
SELECT
    PATIENT AS patient_id,
    ENCOUNTER AS encounter_id,
    CAST(CODE AS STRING) AS condtion_code,
    DESCRIPTION AS condition_desc,
    SYSTEM AS code_system,
    START AS onset_date,
    STOP AS resolved_date,
    (STOP IS NULL ) as is_active,
    DATEDIFF (COALESCE(STOP, current_date()),START) AS duration_days
FROM deduplicated
WHERE rn =1
    AND PATIENT IS NOT NULL
    AND START IS NOT NULL
)
SELECT * FROM cleanded
WHERE resolved_date is NULL OR resolved_date >= onset_date;

In [0]:
-- =========================================================
-- Silver: medications
-- Derives:duration_days (null = ongoing prescription)
-- =========================================================
CREATE OR REPLACE TABLE health_lakehouse.silver.medications AS
WITH deduplicated AS (
    SELECT *, ROW_NUMBER()OVER(
        PARTITION BY PATIENT, ENCOUNTER, CODE, START
        ORDER BY _ingested_at DESC
    ) AS rn
    FROM health_lakehouse.bronze.medications
),
cleaned AS (
    SELECT
        PATIENT AS patient_id,
        ENCOUNTER AS encounter_id,
        PAYER AS payer_id,
        CAST(CODE AS STRING) AS medication_code,
        DESCRIPTION AS medication_desc,
        START AS start_ts,
        STOP AS stop_ts,
        BASE_COST AS base_cost,
        PAYER_COVERAGE AS payer_coverage,
        DISPENSES AS dispenses,
        TOTALCOST AS total_cost,
        CAST(REASONCODE AS STRING) AS reason_code,
        REASONDESCRIPTION AS reason_desc,
        DATEDIFF (STOP,START) AS duration_days,
        (STOP IS NULL) AS is_ongoing
    FROM deduplicated
    WHERE rn =1
        AND PATIENT IS NOT NULL
        AND START IS NOT NULL
    
)
SELECT * FROM cleaned
WHERE stop_ts IS NULL OR stop_ts >= start_ts;

In [0]:
-- =========================================================
-- Silver: procedures
-- =========================================================
CREATE OR REPLACE TABLE health_lakehouse.silver.procedures AS
WITH deduplicated AS(
    SELECT *, ROW_NUMBER()OVER(
        PARTITION BY PATIENT, ENCOUNTER, CODE, START
        ORDER BY _ingested_at DESC
    ) AS rn
    FROM health_lakehouse.bronze.procedures
),
cleaned AS (
    SELECT
        PATIENT AS patient_id,
        ENCOUNTER AS encounter_id,
        CAST(CODE AS STRING) AS procedure_code,
        DESCRIPTION AS procedure_desc,
        SYSTEM AS code_system,
        START AS start_ts,
        STOP AS stop_ts,
        BASE_COST AS base_cost,
        CAST(REASONCODE AS STRING) AS reason_code,
        REASONDESCRIPTION AS reason_desc
    FROM deduplicated
    WHERE rn =1
        AND PATIENT IS NOT NULL
        AND START IS NOT NULL
)
SELECT * FROM cleaned;

In [0]:
-- =========================================================
-- Silver: Observations
-- Tall key-value table. VALUE holds both numeric and text result,
-- TRY_CAST into value_numeric and retain the raw string
-- =========================================================
CREATE OR REPLACE TABLE health_lakehouse.silver.observations AS
WITH deduplicated AS (
    SELECT *, ROW_NUMBER() OVER(
        PARTITION BY PATIENT, ENCOUNTER, CODE, DATE
        ORDER BY _ingested_at DESC
    ) AS rn
    FROM health_lakehouse.bronze.observations
),
cleaned AS (
    SELECT
        PATIENT AS patient_id,
        ENCOUNTER AS encounter_id,
        DATE AS observed_ts,
        CATEGORY AS category,
        CODE AS observation_code,
        DESCRIPTION AS observation_desc,
        VALUE AS value_raw,
        TRY_CAST(VALUE AS DOUBLE) AS value_numeric,
        UNITS AS units,
        TYPE as value_type
    FROM deduplicated
    WHERE rn=1
        AND PATIENT IS NOT NULL
        AND DATE IS NOT NULL
)
SELECT * FROM cleaned;

In [0]:
SELECT observation_code, observation_desc, units, COUNT(*) AS n
FROM health_lakehouse.silver.observations
GROUP BY observation_code, observation_desc, units
ORDER BY n DESC
LIMIT 40;

In [0]:
SELECT observation_code, observation_desc, units, COUNT(*) AS n
FROM health_lakehouse.silver.observations
WHERE LOWER(observation_desc) RLIKE 'glomerular|urea nitrogen|hemoglobin a1c'
GROUP BY observation_code, observation_desc, units
ORDER BY n DESC;